# Model Selection & Training Strategy

The right model at the right stage matters more than picking the "best" model upfront. This note covers the baseline doctrine, the two-tower architecture, GBDT vs DNN tradeoffs, multi-task learning, and retraining strategy.

## What Interviewers Test
- Baseline-first doctrine and why it's non-negotiable
- Two-tower architecture for candidate retrieval
- GBDT vs deep model decision framework
- Multi-task learning: when and why to add auxiliary objectives
- Retraining cadence decisions (daily vs weekly vs continuous)
- Warm-start vs cold-start training tradeoffs

## Baseline-First Doctrine

**Never start with a neural network.**

| Stage | Model | Value |
|---|---|---|
| **Baseline** | Popularity / heuristic rule | Zero training cost; establishes floor; sanity check |
| **v1** | Logistic regression on engineered features | Fast iteration, interpretable, debuggable |
| **v2** | GBDT (LightGBM/XGBoost) | Better nonlinear patterns, handles mixed feature types |
| **v3** | Two-tower + DNN ranker | Scales to millions of items; learned embeddings |
| **v4** | Multi-task learning / fine-tuned LLM | Joint optimization of multiple signals |

> 💡 **Interview Tip:** Interviewers specifically watch whether you propose v3 immediately. Starting with a baseline shows production maturity. Say: *"I'd first implement a popularity ranker to establish a floor, then iterate."*


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
np.random.seed(42)

X, y = make_classification(n_samples=5000, n_features=20, n_informative=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2)

# Baseline: majority class
baseline_auc = roc_auc_score(y_te, np.ones(len(y_te)) * y_tr.mean())

# v1: Logistic regression
lr = LogisticRegression(max_iter=500).fit(X_tr, y_tr)
lr_auc = roc_auc_score(y_te, lr.predict_proba(X_te)[:,1])

# v2: GBDT
gbt = GradientBoostingClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
gbt_auc = roc_auc_score(y_te, gbt.predict_proba(X_te)[:,1])

print(f"Baseline (majority): AUC = {baseline_auc:.3f}")
print(f"v1 Logistic Reg:     AUC = {lr_auc:.3f}")
print(f"v2 GBDT:             AUC = {gbt_auc:.3f}")
print("\nNote: each step adds complexity; justify it with the AUC delta")


## Two-Tower Architecture

```
Query Tower                  Item Tower
(user + context)             (item features)
    ↓                            ↓
  [MLP layers]               [MLP layers]
    ↓                            ↓
 Query embedding   ·dot·   Item embedding
                              ↓
                     Score = dot product
```

**Key properties:**
- Items can be pre-computed offline → fast ANN retrieval
- Works with new items (content features) → solves cold-start
- Dot product score is compatible with FAISS/HNSW for ANN


In [ ]:
import torch
import torch.nn as nn

class TowerNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )
    def forward(self, x):
        emb = self.net(x)
        return emb / (emb.norm(dim=-1, keepdim=True) + 1e-8)  # L2 normalize

class TwoTowerModel(nn.Module):
    def __init__(self, user_dim, item_dim, hidden=64, embed=32):
        super().__init__()
        self.query_tower = TowerNet(user_dim, hidden, embed)
        self.item_tower  = TowerNet(item_dim, hidden, embed)

    def forward(self, user_feat, item_feat):
        q = self.query_tower(user_feat)  # (B, embed_dim)
        v = self.item_tower(item_feat)   # (B, embed_dim)
        return (q * v).sum(dim=-1)       # (B,) — dot product scores

model = TwoTowerModel(user_dim=16, item_dim=12)
B = 32
user_feat = torch.randn(B, 16)
item_feat = torch.randn(B, 12)
scores = model(user_feat, item_feat)
print(f"Two-tower scores shape: {scores.shape}")
print(f"Score range: [{scores.min().item():.3f}, {scores.max().item():.3f}]")
print("Scores are bounded by L2 normalization: [-1, 1]")


## GBDT vs Deep Models

| Criterion | GBDT (LightGBM/XGBoost) | Deep Model (DNN) |
|---|---|---|
| **Feature types** | Mixed (numeric + categorical) | Needs encoding |
| **Training data** | < 100M samples | Works at billions |
| **Feature engineering** | Manual, but interpretable | Learns representations |
| **Serving latency** | < 1ms (CPU) | 5–50ms (GPU or CPU) |
| **Cold-start** | Poor (needs history) | Better (content features) |
| **Hyperparameter tuning** | Easier | Harder, more sensitive |
| **When to use** | Structured data, ranking with features | Embeddings, multi-modal, scale |


## Retraining Cadence

| Signal type | Retraining cadence | Why |
|---|---|---|
| Breaking news / trends | Continuous (hours) | Signal decays within hours |
| User preferences | Daily | Preferences shift slowly |
| Product catalog | Weekly | Catalog changes infrequently |
| Foundational embeddings | Monthly | Expensive; stable |

**Warm-start vs cold-start:**
- **Warm-start:** Initialize from previous checkpoint → faster convergence, risk of forgetting new patterns
- **Cold-start:** Train from scratch → safer but expensive; use for major data distribution shifts


## Common Interview Questions

**Q: Why start with a logistic regression baseline before a neural network?**
Baselines expose data quality issues, establish a performance floor, are fast to iterate, and serve as a sanity check — if a simple model beats a complex one, something is wrong with the training data or features. They also provide interpretability that helps feature engineering.

**Q: When would you choose GBDT over a DNN for ranking?**
When you have well-engineered structured features, a dataset under ~100M rows, latency constraints requiring CPU serving, or interpretability requirements. GBDTs handle categorical features natively, are less sensitive to hyperparameters, and often match DNNs on tabular data.

**Q: What is multi-task learning and when does it help?**
MTL trains one model to optimize multiple objectives simultaneously (e.g., click + watch-time + share). It helps when objectives are correlated (shared representations) and when some tasks have limited labels (transfer learning from related tasks). It risks negative transfer when tasks conflict.

**Q: How do you decide between daily and weekly retraining?**
Monitor prediction distribution drift (PSI) and online metric decay over time. If PSI crosses a threshold (typically 0.2) within 24 hours, move to daily or continuous. If it takes weeks to drift, weekly is sufficient. The cost of retraining must be weighed against the performance degradation from staleness.

## Key Takeaways
- Always start with a non-ML baseline, then logistic regression, then GBDT, then DNN
- Two-tower: separate query and item towers; item embeddings precomputed → fast ANN retrieval
- GBDT: tabular data, lower scale, interpretability; DNN: embeddings, large scale, multi-modal
- Retraining cadence driven by signal freshness and PSI drift monitoring
- Warm-start from checkpoints for efficiency; cold-start for major distribution shifts
- Multi-task learning helps with correlated objectives and label scarcity